# a2_synth

> Phase 21 (`specs/multiagent/phase21/`): synthetic rows and power for A2, kept apart so `a2_report` stays under 200 lines. `syntheticRows21` builds every planned board of the experience screen (8 teams, 3 arms, 6 boards) and of the rule-text screen (P2) with known effects, for the offline tests of `a2Summaries`; `pilotSigma` reads the spread of per-identity guarding from the pilot's boards 3-6; `power21` simulates the power of Gate A2's sign-flip test and of H21-2 from that spread, before the main run.

In [ ]:
#| default_exp a2_synth

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json

import numpy as np

from explRL.a2_cols import HONEST, LIARS, failureIds, successIds
from explRL.a2_grid import A2_PLAN, P2_N, p2Grid
from explRL.a2_seq import _key
from explRL.colony_report import signFlipExact

EFFECTS21 = dict(own=0.2, shared=0.0, none=0.0, p2=-0.3)   # success - failure guarding on boards 3-6; P2: drop - orig suppressedHeld


def _seqRow(rng, team, arm, b, eff, manip, noise):
    "One synthetic sequence board with every column a2Summaries reads (identity order = slot order after the 4 liars)."
    N, base = LIARS + HONEST, np.random.default_rng([team, 7]).uniform(0.05, 0.3, HONEST)
    g = base + rng.normal(0, noise, HONEST)
    if b >= 3:
        g[successIds(team)] += eff / 2
        g[failureIds(team)] -= eff / 2
    g = np.clip(g, 0, 1)
    con = rng.poisson(1 + 4 * base)
    fol = np.minimum(con, rng.poisson(0.5, HONEST))
    if b <= 2:
        con[successIds(team)], fol[successIds(team)] = np.maximum(con[successIds(team)], 1), int(manip)
    turns, sh = [40] * N, [0] * LIARS + [int(x) for x in np.round(40 * g)]
    sub = [0] * LIARS + [int(x) for x in rng.integers(2, 8, HONEST)]
    j = lambda xs: json.dumps([int(x) if isinstance(x, (int, np.integer)) else float(x) for x in xs])
    return dict(study='a2', level=('common' if b <= 2 else arm) + f'|b{b}', arm=arm, seqId=team, pos=b - 1, seed=10 * team + b, N=N, nLiar=LIARS,
                idGuarding=j(g), idContests=j(con), idFollowed=j(fol), liarOnFailure=0, agents=j(range(N)), agentTurns=j(turns),
                agentShares=j(np.maximum(sh, [0] * LIARS + list(con))), agentContests=j([0] * LIARS + list(con)), agentSubmits=j(sub),
                agentOpens=j([0] * LIARS + [3] * HONEST), contestLags=j(rng.integers(1, 6, 3)), misinformedFirst=int(rng.integers(0, 5)),
                honestTurns=480, uncorrected=int(rng.integers(0, 4)), liarFirst=6, contests=int(con.sum()), liarPosts=80, guardLies=20,
                sameWrong=1, wrongHonest=6, sharePerHeld=float(rng.uniform(0.4, 0.8)), welfare=float(rng.uniform(0.05, 0.1)),
                welfareHonest=float(rng.uniform(0.05, 0.12)), tEnd=40.0, inputTokens=2.0e6, outputTokens=1.0e5, solved=20,
                volunteers=int(rng.integers(0, 2)), specIndex=float(rng.uniform(1, 2)), specNull=1.0, invalid=0, calls=480, cost=0.2)


def syntheticRows21(effects=None, teams=None, plan=A2_PLAN, noise=0.02, manipTeams=None, seed=0):
    """(rows, p2rows): every planned board with known effects (EFFECTS21 by default); boards 1-2 copied into each arm as
    runSequences21 does; manipTeams: how many teams pass the manipulation check (default all)."""
    e, rng = dict(EFFECTS21, **(effects or {})), np.random.default_rng(seed)
    teams = teams or plan['teams']
    nm = len(teams) if manipTeams is None else manipTeams
    rows = []
    for k, t in enumerate(teams):
        for b in range(1, plan['boards'] + 1):
            if b <= 2:
                r = _seqRow(rng, t, None, b, 0.0, k < nm, noise)
                rows += [dict(r, arm=a) for a in plan['arms']]
            else:
                rows += [_seqRow(rng, t, a, b, e[a], True, noise) for a in plan['arms']]
    p2 = []
    for c in p2Grid(plan['p2']['seeds']):
        drop = c['study'] == 'a2drop'
        sh = float(np.clip(0.9 + (e['p2'] if drop else 0.0) + rng.normal(0, 0.05), 0, 1))
        p2.append(dict(study=c['study'], level=c['level'], N=c['N'], seed=c['seed'], nLiar=c['N'] // 4, suppressedHeld=sh, suppressed=sh,
                       firstWrongPerTurn=0.02, contaminated=0.3, honestPooled=0.5, sharePerHeld=0.6, misinformedFirst=3,
                       honestTurns=30 * c['N'], uncorrected=5, liarFirst=6, contests=2, liarPosts=40, guardLies=0, sameWrong=1,
                       wrongHonest=5, contestLags='[2]', welfare=0.05, welfareHonest=0.06, tEnd=40.0, inputTokens=1e6,
                       outputTokens=5e4, solved=10, invalid=0, calls=30 * c['N'], cost=0.1))
    return rows, p2


def pilotSigma(rows):
    "The SD of per-identity guarding across honest identities and boards 3-6 of the pilot (its spread around each board's mean)."
    d = [x - np.nanmean(g) for r in rows if _key(r)[2] >= 3 for g in [np.array(json.loads(r['idGuarding']), float)] for x in g if np.isfinite(x)]
    return float(np.std(d, ddof=1)) if len(d) > 1 else float('nan')


def power21(sigma, effects=(0.05, 0.1, 0.2, 0.3), teams=8, nsim=2000, alpha=0.05, seed=0):
    """Power of Gate A2 (gate statistic SD sigma / sqrt(2): two arms, 4 boards, pairs of 2) and of H21-2 (SD sigma / 2) by
    simulation with the exact sign-flip test, for success - failure effects on guarding."""
    rng, out = np.random.default_rng(seed), {}
    for name, sd in (('gate', sigma / np.sqrt(2)), ('H21_2', sigma / 2)):
        out[name] = {d: float(np.mean([signFlipExact(rng.normal(d, sd, teams)) <= alpha for _ in range(nsim)])) for d in effects}
    return dict(sigma=float(sigma), sdGate=float(sigma / np.sqrt(2)), power=out, teams=teams, nsim=nsim)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()